# 01_exploration_francetravail

Les URL (FRANCE TRAVAIL) utiles sont :
- https://francetravail.io/data/api/offres-emploi
- https://francetravail.io/data/api/offres-emploi/documentation#/

In [62]:
import os
import requests
from dotenv import load_dotenv
from pprint import pprint

import json
from rich.tree import Tree
from rich import print as rprint

# Charge le .env en mémoire
load_dotenv()

True

In [63]:
##################  VARIABLES  ##################
# France Travail
FRANCETRAVAIL_CLIENT_ID     = os.getenv("FRANCETRAVAIL_CLIENT_ID")
FRANCETRAVAIL_CLIENT_SECRET = os.getenv("FRANCETRAVAIL_CLIENT_SECRET")

In [64]:
# Vérification de l'accès aux variables relatives à la connexion à l'API de France Travail
variables = [
    "FRANCETRAVAIL_CLIENT_ID",
    "FRANCETRAVAIL_CLIENT_SECRET",
]

for var in variables:
    valeur = os.getenv(var)
    print(f"{var} : {'✅ chargée' if valeur else '❌ manquante'}")

FRANCETRAVAIL_CLIENT_ID : ✅ chargée
FRANCETRAVAIL_CLIENT_SECRET : ✅ chargée


In [65]:
# ---------------------------
# AUTH FRANCE TRAVAIL
# ---------------------------

def get_token(retries=3, wait=5):
    """Récupère un token d'authentification OAuth2."""
    url = "https://entreprise.francetravail.fr/connexion/oauth2/access_token"
    params = {"realm": "/partenaire"}
    data = {
        "grant_type":    "client_credentials",
        "client_id":     FRANCETRAVAIL_CLIENT_ID,
        "client_secret": FRANCETRAVAIL_CLIENT_SECRET,
        "scope":         "api_offresdemploiv2 o2dsoffre"
    }
    for attempt in range(retries):
        try:
            response = requests.post(url, params=params, data=data)
            return response.json()["access_token"]
        except requests.RequestException as e:
                logging.warning(f"Erreur OAuth attempt {attempt+1}: {e}")
                time.sleep(wait)
    raise RuntimeError("Impossible d'obtenir un token OAuth après plusieurs essais.")

In [66]:
# ---------------------------
# REQUETE API FRANCE TRAVAIL
# ---------------------------
def rechercher_offres(token, mots_cles="data engineer", nb_resultats=100):
    """Recherche des offres d'emploi via l'API FranceTravail."""
    url = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept":        "application/json"
    }
    params = {
        "motsCles":   mots_cles,
        "range":      f"0-{nb_resultats - 1}",
        "sort":       "1"
    }

    try: 
        response = requests.get(url, headers=headers, params=params)
        return response.json()
    except requests.RequestException as e:
        print(f"Erreur API France Travail: {e}")

In [67]:
# ---------------------------
# REQUETE API FRANCE TRAVAIL
# ---------------------------

def json_vers_arbre(data, arbre, prefixe=""):
    """
    Construit récursivement un arbre Rich
    à partir d'un dictionnaire JSON.
    """
    if isinstance(data, dict):
        for cle, valeur in data.items():
            if isinstance(valeur, dict):
                # Sous-dictionnaire → nouveau nœud
                noeud = arbre.add(f"[bold cyan]{cle}[/] 📁")
                json_vers_arbre(valeur, noeud)

            elif isinstance(valeur, list):
                # Tableau → nouveau nœud avec indication du type
                noeud = arbre.add(f"[bold yellow]{cle}[/] 📋 [dim]({len(valeur)} éléments)[/]")
                if valeur and isinstance(valeur[0], dict):
                    json_vers_arbre(valeur[0], noeud)

            else:
                # Valeur simple → feuille
                type_valeur = type(valeur).__name__
                arbre.add(f"[green]{cle}[/] [dim]({type_valeur})[/]")


def afficher_structure_json(data, titre="Structure JSON"):
    """
    Affiche la structure d'un JSON sous forme d'arbre visuel.
    """
    arbre = Tree(f"[bold magenta]{titre}[/]")
    json_vers_arbre(data, arbre)
    rprint(arbre)

In [72]:
if __name__ == "__main__":
    
    # Récupération token France travail
    token  = get_token()

    # Requête vers API france Travail
    offres = rechercher_offres(token, "data engineer", nb_resultats = 1)
    print(f"{len(offres.get('resultats', []))} offres récupérées") 

    # Afficher 
    afficher_structure_json(offres, "Offre FranceTravail")
    pprint(offres)

1 offres récupérées


Offre FranceTravail
├── resultats 📋 (1 éléments)
│   ├── id (str)
│   ├── intitule (str)
│   ├── description (str)
│   ├── dateCreation (str)
│   ├── dateActualisation (str)
│   ├── lieuTravail 📁
│   │   ├── libelle (str)
│   │   ├── latitude (float)
│   │   ├── longitude (float)
│   │   ├── codePostal (str)
│   │   └── commune (str)
│   ├── romeCode (str)
│   ├── romeLibelle (str)
│   ├── appellationlibelle (str)
│   ├── entreprise 📁
│   │   └── description (str)
│   ├── typeContrat (str)
│   ├── typeContratLibelle (str)
│   ├── natureContrat (str)
│   ├── experienceExige (str)
│   ├── experienceLibelle (str)
│   ├── salaire 📁
│   ├── dureeTravailLibelle (str)
│   ├── alternance (bool)
│   ├── contact 📁
│   ├── nombrePostes (int)
│   ├── origineOffre 📁
│   │   ├── origine (str)
│   │   ├── urlOrigine (str)
│   │   └── partenaires 📋 (1 éléments)
│   │       ├── nom (str)
│   │       ├── url (str)
│   │       └── logo (str)
│   ├── contexteTravail 📁
│   │   └── horaires 📋 (1 éléments)
│   ├── entrepriseAdaptee (bool)
│   └── employeurHandiEngage (bool)
└── filtresPossibles 📋 (4 éléments)
    ├── filtre (str)
    └── agregation 📋 (4 éléments)
        ├── valeurPossible (str)
        └── nbResultats (int)

{'filtresPossibles': [{'agregation': [{'nbResultats': 62,
                                       'valeurPossible': 'CDD'},
                                      {'nbResultats': 388,
                                       'valeurPossible': 'CDI'},
                                      {'nbResultats': 7,
                                       'valeurPossible': 'LIB'},
                                      {'nbResultats': 58,
                                       'valeurPossible': 'MIS'}],
                       'filtre': 'typeContrat'},
                      {'agregation': [{'nbResultats': 72,
                                       'valeurPossible': '0'},
                                      {'nbResultats': 190,
                                       'valeurPossible': '4'},
                                      {'nbResultats': 36,
                                       'valeurPossible': '1'},
                                      {'nbResultats': 141,
                                   